# Bielik on TPU — Episode 03: Profiling & HLO (trace.json.gz and hlo.txt)

Companion notebook for the [`bielik-on-tpu`](https://github.com/qooba/bielik-on-tpu) repo's YT series.

In episodes 01 and 02, we measured performance using wall-clock time (`time.perf_counter` synchronized with `block_until_ready`). While great for a quick look, wall-clock timing lumps host dispatch and device execution together.

In this notebook, we take our two kernels (**RMSNorm** and **Matmul**) and go down to the metal using two diagnostic tools:
1. **`jax.profiler` (`trace.json.gz`)**: Peeking inside runtime traces to isolate host dispatch from pure TPU execution, inspecting Chrome Trace Event JSON schemas, and fixing the Pallas custom-call blind spot using `cost_estimate`.
2. **Compiled HLO IR (`hlo.txt`)**: Exporting post-optimization High-Level Optimizer text (`.compile().as_text()`) to verify what XLA actually fuses under the hood vs Pallas's opaque custom-call boundary.

**Hardware requirement**: `Runtime` -> `Change runtime type` -> select **TPU**. Real Pallas Mosaic lowering requires a real TPU.

## 1. Clone the repository

`qooba/bielik-on-tpu` is public — no credentials needed.

In [ ]:
import os

REPO_URL = "https://github.com/qooba/bielik-on-tpu.git"
REPO_DIR = "/content/bielik-on-tpu"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 2. Install the repo as an editable package

`pyproject.toml` declares `bielik`/`kernels`/`benchmarks`/`profiling` as an installable package with no pinned dependencies of its own.

In [ ]:
!pip install -e . --no-deps -q

## 3. Fix the preinstalled `jax`/`libtpu` version mismatch

Colab's TPU base image can have a mismatched `libtpu`. We reinstall the pinned pair and restart the runtime.

Running the next cell **restarts the Colab runtime**. After it restarts, continue from the verification cell below.

In [ ]:
!pip install -q -U 'jax[tpu]==0.11.1' -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

import os

os.kill(os.getpid(), 9)  # restart runtime so new libtpu loads

### Verify TPU Accelerator

In [ ]:
%cd /content/bielik-on-tpu

import jax

print("Visible JAX devices:", jax.devices())
assert any(
    "TPU" in d.device_kind for d in jax.devices()
), "No TPU device found! Switch runtime to TPU."

## 4. Import the kernels

We import the RMSNorm and matmul reference + Pallas implementations from episodes 1 and 2.

In [ ]:
import gzip
import json
import shutil
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np

from kernels.matmul.autotune import autotune_matmul, matmul_pallas_autotuned
from kernels.matmul.matmul import matmul_pallas, matmul_ref
from kernels.normalization.autotune import autotune_rms_norm
from kernels.normalization.rms_norm import rms_norm_pallas, rms_norm_ref

## 5. Capturing a trace with `jax.profiler.trace`

In a headless environment, profiling takes just a few lines of Python:
1. Run a warmup call with `jax.block_until_ready` to avoid measuring one-time JIT compilation overhead.
2. Wrap steady-state execution inside `jax.profiler.trace(trace_dir)`.
3. Wrap each invocation in a `jax.profiler.TraceAnnotation(name)` to tag host-side spans.

In [ ]:
TRACE_DIR = Path("/tmp/profile_trace_demo")
if TRACE_DIR.exists():
    shutil.rmtree(TRACE_DIR)
TRACE_DIR.mkdir(parents=True)

key_x, key_w = jax.random.split(jax.random.PRNGKey(0))
x = jax.random.normal(key_x, (4096, 4096), dtype=jnp.bfloat16)
w = jax.random.normal(key_w, (4096,), dtype=jnp.bfloat16)

rms_norm_ref_jit = jax.jit(rms_norm_ref)
rms_norm_pallas_jit = jax.jit(rms_norm_pallas, static_argnames=("eps", "block_rows", "interpret"))

n_rows, hidden_size = x.shape
best_block_rows = autotune_rms_norm(n_rows, hidden_size, dtype=x.dtype)

jax.block_until_ready(rms_norm_ref_jit(x, w))
jax.block_until_ready(rms_norm_pallas_jit(x, w, block_rows=best_block_rows))

native_dir = TRACE_DIR / "native"
with jax.profiler.trace(str(native_dir)):
    for _ in range(5):
        with jax.profiler.TraceAnnotation("rms_norm_ref"):
            jax.block_until_ready(rms_norm_ref_jit(x, w))

pallas_dir = TRACE_DIR / "pallas"
with jax.profiler.trace(str(pallas_dir)):
    for _ in range(5):
        with jax.profiler.TraceAnnotation("rms_norm_pallas_autotuned"):
            jax.block_until_ready(rms_norm_pallas_jit(x, w, block_rows=best_block_rows))

native_trace_file = next(native_dir.rglob("*.trace.json.gz"))
pallas_trace_file = next(pallas_dir.rglob("*.trace.json.gz"))
print("Native trace captured at:", native_trace_file)
print(f"Pallas (autotuned, block_rows={best_block_rows}) trace captured at:", pallas_trace_file)

## 6. Dissecting the native Chrome Trace Event Format

`trace.json.gz` is a standard gzipped JSON file conforming to Google's **Chrome Trace Event Format** (the exact format Perfetto and `chrome://tracing` read).

Let's decompress and inspect its top-level structure and event phases (`ph`):
* **`ph: "M"` (Metadata Events)**: Configures system hierarchy without measuring time (`process_name`, `thread_name`). Tells us which `pid` is `/device:TPU:0` and which `tid` is `XLA Ops`.
* **`ph: "X"` (Complete Events)**: Represents an atomic execution block on the hardware, carrying both start timestamp `ts` and execution duration `dur` in microseconds!

In [ ]:
with gzip.open(native_trace_file) as f:
    native_data = json.load(f)

print("Top-level JSON keys:", list(native_data.keys()))
print("Display time unit:", native_data.get("displayTimeUnit"))
events = native_data["traceEvents"]
print(f"Total events in trace: {len(events)}")

# Inspect metadata events (ph == 'M')
meta_events = [e for e in events if e.get("ph") == "M"]
print(f"\nMetadata events (ph == 'M'): {len(meta_events)}")

processes = {e["pid"]: e["args"]["name"] for e in meta_events if e.get("name") == "process_name"}
print("Processes discovered:")
for pid, name in processes.items():
    print(f"  PID {pid}: {name}")

device_pid = next(pid for pid, name in processes.items() if "TPU" in name)
print(f"\nIdentified TPU device PID: {device_pid} ({processes[device_pid]})")

## 7. Extracting and comparing on-chip TPU operations

We filter for completed events (`ph == "X"`) running on `device_pid` with an HLO category.

Let's see what actually executed on the TPU for both JAX Native and Pallas!

In [ ]:
def extract_ops(trace_path: Path):
    with gzip.open(trace_path) as f:
        data = json.load(f)
    evs = data["traceEvents"]
    dev_pid = next(
        e["pid"]
        for e in evs
        if e.get("ph") == "M" and e.get("name") == "process_name" and "TPU" in e["args"]["name"]
    )
    return [
        e
        for e in evs
        if e.get("pid") == dev_pid
        and e.get("ph") == "X"
        and e.get("args", {}).get("hlo_category")
        in ["loop fusion", "custom-call", "convolution fusion"]
    ]


native_ops = extract_ops(native_trace_file)
pallas_ops = extract_ops(pallas_trace_file)


def extract_substantial_ops(trace_path: Path, min_dur_us: float = 1.0):
    with gzip.open(trace_path) as f:
        data = json.load(f)
    evs = data["traceEvents"]
    dev_pid = next(
        e["pid"]
        for e in evs
        if e.get("ph") == "M" and e.get("name") == "process_name" and "TPU" in e["args"]["name"]
    )
    return [
        e
        for e in evs
        if e.get("pid") == dev_pid
        and e.get("ph") == "X"
        and e.get("args", {}).get("hlo_category")
        and e["dur"] > min_dur_us
    ]


native_substantial = extract_substantial_ops(native_trace_file)
pallas_substantial = extract_substantial_ops(pallas_trace_file)


def average_by_name(ops):
    order, totals, counts, categories = [], {}, {}, {}
    for op in ops:
        name = op["name"]
        if name not in totals:
            order.append(name)
            totals[name] = 0.0
            counts[name] = 0
            categories[name] = op["args"].get("hlo_category")
        totals[name] += op["dur"]
        counts[name] += 1
    return [(name, totals[name] / counts[name], counts[name], categories[name]) for name in order]


print("=== JAX Native Operations on TPU (thread: XLA Ops, averaged over reps) ===")
native_avg = average_by_name(native_substantial)
for name, avg_dur, n, category in native_avg:
    print(f"  {name:<20} {avg_dur:>7.2f} us  [{category}]  (avg of {n} reps)")
native_subtotal = sum(avg_dur for _, avg_dur, _, _ in native_avg)
print(f"  --> Total on TPU (one call): {native_subtotal:.2f} us")

print("\n=== Pallas Operations on TPU (thread: XLA Ops, averaged over reps) ===")
pallas_avg = average_by_name(pallas_substantial)
for name, avg_dur, n, category in pallas_avg:
    print(f"  {name:<20} {avg_dur:>7.2f} us  [{category}]  (avg of {n} reps)")

## 8. The Pallas blind spot in event args

Let's print the full event dictionary for both kernels. Look closely at the `args` field:
* In **JAX Native**, XLA's internal cost model calculates `bytes_accessed` and `model_flops`, and tags the exact source line (`kernels/normalization/rms_norm.py`).
* In **Pallas**, because it compiles to an opaque `custom-call`, XLA cannot see inside our tile loops. Its cost model gives up and reports `bytes_accessed: 0` and `model_flops: 0`!

In [ ]:
# Pick the normalization fusion
sample_native_op = next(op for op in native_ops if "fusion" in op["name"])

print("--- JAX Native Event JSON ---")
print(
    json.dumps(
        {
            "name": sample_native_op["name"],
            "dur": round(sample_native_op["dur"], 2),
            "args": {
                "hlo_category": sample_native_op["args"].get("hlo_category"),
                "bytes_accessed": sample_native_op["args"].get("bytes_accessed"),
                "model_flops": sample_native_op["args"].get("model_flops"),
                "shape_with_layout": sample_native_op["args"].get("shape_with_layout"),
                "source": sample_native_op["args"].get("source"),
            },
        },
        indent=2,
    )
)

print("\n--- Pallas Custom-Call Event JSON ---")
print(
    json.dumps(
        {
            "name": pallas_ops[0]["name"],
            "dur": round(pallas_ops[0]["dur"], 2),
            "args": {
                "hlo_category": pallas_ops[0]["args"].get("hlo_category"),
                "bytes_accessed": pallas_ops[0]["args"].get("bytes_accessed"),
                "model_flops": pallas_ops[0]["args"].get("model_flops"),
                "source": pallas_ops[0]["args"].get("source"),
            },
        },
        indent=2,
    )
)

## 9. Case Study: The Host-vs-Device time gap in RMSNorm

In episode 01, our wall-clock benchmark showed that Pallas was struggling to beat JAX Native on smaller tensors, only pulling ahead at scale (4096).

Here is why:
* **Host-side dispatch** (`TraceAnnotation` span) carries a relatively flat latency (~150-250 µs) of Python and asynchronous runtime scheduling.
* **Device compute** (`dur`) scales with tensor data volume.

Let's measure Host wall time vs pure Device time across a square size sweep (`512` to `4096`).

In [ ]:
import pandas as pd

sizes = [512, 1024, 2048, 4096]
rows = []

for size in sizes:
    kx, kw = jax.random.split(jax.random.PRNGKey(size))
    xt = jax.random.normal(kx, (size, size), dtype=jnp.bfloat16)
    wt = jax.random.normal(kw, (size,), dtype=jnp.bfloat16)

    jax.block_until_ready(rms_norm_pallas_jit(xt, wt))

    sdir = TRACE_DIR / f"sweep_{size}"
    with jax.profiler.trace(str(sdir)):
        for _ in range(5):
            with jax.profiler.TraceAnnotation(f"pallas_{size}"):
                jax.block_until_ready(rms_norm_pallas_jit(xt, wt))

    st = extract_ops(next(sdir.rglob("*.trace.json.gz")))
    dev_us = st[0]["dur"]

    with gzip.open(next(sdir.rglob("*.trace.json.gz"))) as f:
        all_evs = json.load(f)["traceEvents"]
    host_evs = [e for e in all_evs if e.get("name") == f"pallas_{size}"]
    host_us = host_evs[-1]["dur"] if host_evs else np.nan

    rows.append(
        {
            "Tensor Size": f"{size}x{size}",
            "Device TPU Time (us)": round(dev_us, 2),
            "Host Dispatch (us)": round(host_us, 2),
            "Host / Device Ratio": f"{host_us / dev_us:.2f}x",
        }
    )

df = pd.DataFrame(rows)
display(df)

## 10. Fixing the blind spot with `cost_estimate`

To restore throughput metrics (GB/s and TFLOPS) for Pallas, we pass `cost_estimate` to `pallas_call`:
* **Manual pattern** (Matmul): exact FLOP and byte formulas.
* **Auto-derived pattern** (RMSNorm): `pl.estimate_cost(rms_norm_ref, x, w, eps=eps)`.

Let's verify that `cost_estimate`:
1. Produces identical outputs.
2. Incurs zero runtime overhead.
3. Successfully populates `bytes_accessed` and `model_flops` in the trace!

In [ ]:
import functools

from jax.experimental import pallas as pl

from kernels.normalization.rms_norm import _rms_norm_kernel


def rms_norm_pallas_with_cost_estimate(x, w, eps=1e-6, block_rows=8):
    # pl.estimate_cost traces rms_norm_ref to auto-derive flops/transcendentals --
    # must pass eps=eps (not positionally): pl.estimate_cost binds **kwargs via
    # functools.partial before tracing *args, so a positional eps crashes with
    # AttributeError: 'float' object has no attribute 'shape'.
    n_rows, hidden_size = x.shape
    body_cost = pl.estimate_cost(rms_norm_ref, x, w, eps=eps)
    cost_estimate = pl.CostEstimate(
        flops=body_cost.flops,
        transcendentals=body_cost.transcendentals,
        bytes_accessed=2 * x.size * x.dtype.itemsize,
    )

    kernel = functools.partial(_rms_norm_kernel, eps=eps)
    return pl.pallas_call(
        kernel,
        grid=(pl.cdiv(n_rows, block_rows),),
        in_specs=[
            pl.BlockSpec((block_rows, hidden_size), lambda i: (i, 0)),
            pl.BlockSpec((hidden_size,), lambda i: (0,)),
        ],
        out_specs=pl.BlockSpec((block_rows, hidden_size), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        cost_estimate=cost_estimate,
    )(x, w)


rms_norm_pallas_with_cost_estimate_jit = jax.jit(
    rms_norm_pallas_with_cost_estimate, static_argnames=("eps", "block_rows")
)

out_plain = rms_norm_pallas(x, w)
out_cost = rms_norm_pallas_with_cost_estimate_jit(x, w)

diff = float(jnp.max(jnp.abs(out_plain.astype(jnp.float32) - out_cost.astype(jnp.float32))))
print("Max absolute difference (with vs without cost_estimate):", diff)
assert diff == 0.0, "cost_estimate must not modify computation!"

cost_dir = TRACE_DIR / "cost_estimate"
with jax.profiler.trace(str(cost_dir)):
    for _ in range(5):
        with jax.profiler.TraceAnnotation("pallas_cost"):
            jax.block_until_ready(rms_norm_pallas_with_cost_estimate_jit(x, w))

cost_ops = extract_ops(next(cost_dir.rglob("*.trace.json.gz")))
op = cost_ops[0]

dur_us = op["dur"]
bytes_acc = float(op["args"].get("bytes_accessed", 0))
flops = float(op["args"].get("model_flops", 0))

gbps = (bytes_acc / (dur_us * 1e-6)) / 1e9
tflops = (flops / (dur_us * 1e-6)) / 1e12

peak_bw = 800.0 * (2**30) / 1e9  # TPU v5e: 800 GiB/s

print(f"Device Duration:    {dur_us:.2f} us")
print(f"Bytes Accessed:     {int(bytes_acc):,}")
print(f"Model FLOPs:        {int(flops):,}")
print(f"Achieved Memory BW: {gbps:.2f} GB/s ({gbps / peak_bw * 100:.1f}% of TPU v5e HBM peak)")
print(f"Achieved Compute:   {tflops:.4f} TFLOPS")

## 11. Case Study: Matmul autotuning verification on hardware

In episode 02, autotuning increased matmul throughput dramatically. Let's verify this directly on the TPU device at $M=K=N=2048$.

In [ ]:
from jax.experimental.pallas import tpu as pltpu

from kernels.matmul.matmul import _matmul_bias_kernel, _matmul_kernel


def _matmul_cost_estimate(a, b, bias, M, K, N):
    itemsize = a.dtype.itemsize
    flops = 2 * M * K * N
    bytes_accessed = (a.size + b.size + M * N) * itemsize
    if bias is not None:
        flops += M * N
        bytes_accessed += bias.size * itemsize
    return pl.CostEstimate(flops=flops, transcendentals=0, bytes_accessed=bytes_accessed)


def matmul_pallas_with_cost_estimate(a, b, bias=None, block_m=128, block_n=128, block_k=128):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, f"inner dimensions must match: {K} != {K2}"

    k_steps = pl.cdiv(K, block_k)
    grid = (pl.cdiv(M, block_m), pl.cdiv(N, block_n), k_steps)
    in_specs = [
        pl.BlockSpec((block_m, block_k), lambda i, j, k: (i, k)),
        pl.BlockSpec((block_k, block_n), lambda i, j, k: (k, j)),
    ]
    out_specs = pl.BlockSpec((block_m, block_n), lambda i, j, k: (i, j))
    out_shape = jax.ShapeDtypeStruct((M, N), a.dtype)
    scratch_shapes = [pltpu.VMEM((block_m, block_n), jnp.float32)]
    cost_estimate = _matmul_cost_estimate(a, b, bias, M, K, N)

    if bias is None:
        kernel = functools.partial(_matmul_kernel, k_steps=k_steps, block_k=block_k, K=K)
        return pl.pallas_call(
            kernel,
            grid=grid,
            in_specs=in_specs,
            out_specs=out_specs,
            out_shape=out_shape,
            scratch_shapes=scratch_shapes,
            cost_estimate=cost_estimate,
        )(a, b)

    bias_2d = bias.reshape(1, N)
    bias_spec = pl.BlockSpec((1, block_n), lambda i, j, k: (0, j))
    kernel = functools.partial(_matmul_bias_kernel, k_steps=k_steps, block_k=block_k, K=K)
    return pl.pallas_call(
        kernel,
        grid=grid,
        in_specs=[*in_specs, bias_spec],
        out_specs=out_specs,
        out_shape=out_shape,
        scratch_shapes=scratch_shapes,
        cost_estimate=cost_estimate,
    )(a, b, bias_2d)


matmul_pallas_with_cost_estimate_jit = jax.jit(
    matmul_pallas_with_cost_estimate, static_argnames=("block_m", "block_n", "block_k")
)

M, K, N = 2048, 2048, 2048
ka, kb = jax.random.split(jax.random.PRNGKey(1))
a = jax.random.normal(ka, (M, K), dtype=jnp.bfloat16)
b = jax.random.normal(kb, (K, N), dtype=jnp.bfloat16)

matmul_ref_jit = jax.jit(matmul_ref)
matmul_pallas_jit = jax.jit(
    matmul_pallas, static_argnames=("block_m", "block_n", "block_k", "interpret")
)

best_block_m, best_block_n, best_block_k = autotune_matmul(M, K, N, dtype=a.dtype, bias=False)

jax.block_until_ready(matmul_ref_jit(a, b))
jax.block_until_ready(
    matmul_pallas_with_cost_estimate_jit(a, b, block_m=128, block_n=128, block_k=128)
)
jax.block_until_ready(
    matmul_pallas_jit(a, b, None, best_block_m, best_block_n, best_block_k, False)
)

matmul_trace_dir = TRACE_DIR / "matmul"

with jax.profiler.trace(str(matmul_trace_dir / "fixed128")):
    for _ in range(5):
        jax.block_until_ready(
            matmul_pallas_with_cost_estimate_jit(a, b, block_m=128, block_n=128, block_k=128)
        )

with jax.profiler.trace(str(matmul_trace_dir / "autotuned")):
    for _ in range(5):
        jax.block_until_ready(
            matmul_pallas_jit(a, b, None, best_block_m, best_block_n, best_block_k, False)
        )

with jax.profiler.trace(str(matmul_trace_dir / "native")):
    for _ in range(5):
        jax.block_until_ready(matmul_ref_jit(a, b))

fixed_op = extract_ops(next((matmul_trace_dir / "fixed128").rglob("*.trace.json.gz")))[0]
autotuned_op = extract_ops(next((matmul_trace_dir / "autotuned").rglob("*.trace.json.gz")))[0]
native_ops_raw = extract_ops(next((matmul_trace_dir / "native").rglob("*.trace.json.gz")))
# native_ops_raw also contains zero-flops copy-start/copy-done bookkeeping ops --
# pick the op with nonzero model_flops specifically, not just the first entry.
native_compute_ops = [op for op in native_ops_raw if float(op["args"].get("model_flops", 0)) > 0]
native_op = max(native_compute_ops, key=lambda op: op["dur"])

# autotuned_op has no cost_estimate attached (it's the plain matmul_pallas_jit) --
# fall back to the exact manual formula, same FLOP count regardless of which
# block config won.
flops_2048 = 2 * M * K * N

fixed_tflops = (float(fixed_op["args"]["model_flops"]) / (fixed_op["dur"] * 1e-6)) / 1e12
autotuned_tflops = (flops_2048 / (autotuned_op["dur"] * 1e-6)) / 1e12
native_tflops = (float(native_op["args"]["model_flops"]) / (native_op["dur"] * 1e-6)) / 1e12

peak_tflops = 197.0  # TPU v5e bf16 peak

rows = [
    ("Pallas (Fixed Block 128)", fixed_op["dur"], fixed_tflops),
    (
        f"Pallas (Autotuned {best_block_m}/{best_block_n}/{best_block_k})",
        autotuned_op["dur"],
        autotuned_tflops,
    ),
    ("JAX Native (Reference)", native_op["dur"], native_tflops),
]
print(f"{'Provider':<40} {'TPU Time':>12} {'Device TFLOPS':>15} {'% of TPU v5e Peak':>18}")
for name, dur_us, tflops in rows:
    print(
        f"{name:<40} {dur_us / 1e3:>10.2f} ms {tflops:>13.2f} {tflops / peak_tflops * 100:>16.1f}%"
    )

## 12. Tool 2: Peeking into the compiler brain (`hlo.txt`)

Runtime traces tell us *when* and *how fast* things ran. But they cannot answer: *what did XLA build structurally?*

To inspect the compiler IR:
1. `jitted_fn.lower(*args)`: translates Python to unoptimized HLO.
2. `.compile()`: runs XLA optimization, buffer assignment, and **operator fusion** passes!
3. `.as_text()`: exports the post-optimization HLO representation.

In [ ]:
jitted_native = jax.jit(rms_norm_ref)
lowered = jitted_native.lower(x, w)
compiled = lowered.compile()

hlo_text = compiled.as_text()
hlo_path = Path("/tmp/rms_norm_native.hlo.txt")
hlo_path.write_text(hlo_text)

print(
    f"Exported HLO to {hlo_path} ({len(hlo_text):,} characters, {len(hlo_text.splitlines())} lines)"
)
print("\nFirst 25 lines of compiled HLO:\n")
for line in hlo_text.splitlines()[:25]:
    print(line)

## 13. Structural fusion analysis: XLA fusion vs Pallas custom-call

What happens when we chain an operation — e.g., Matmul followed immediately by a residual addition (`x @ w + residual`)?
* In **JAX Native**: XLA inspects the graph and merges dot product and addition into a single `%fusion` instruction! The intermediate matrix is never written back to HBM.
* In **Pallas**: `pallas_call` lowers to an opaque `custom-call`. XLA cannot peer inside, so the addition remains a separate, standalone `add` instruction!

In [ ]:
def native_chain(x, w, res):
    return matmul_ref(x, w) + res


def pallas_chain(x, w, res):
    return matmul_pallas_autotuned(x, w) + res


M_sub, K_sub, N_sub = 128, 1536, 1536
xa = jax.random.normal(key_x, (M_sub, K_sub), dtype=jnp.bfloat16)
wa = jax.random.normal(key_w, (K_sub, N_sub), dtype=jnp.bfloat16)
ra = jax.random.normal(key_x, (M_sub, N_sub), dtype=jnp.bfloat16)

# Compile both chains to HLO
hlo_native = jax.jit(native_chain).lower(xa, wa, ra).compile().as_text()
hlo_pallas = jax.jit(pallas_chain).lower(xa, wa, ra).compile().as_text()

print("=== JAX Native HLO (Fusion inspection) ===")
fusion_lines = [
    line for line in hlo_native.splitlines() if "fusion(" in line or "%fused_computation" in line
]
for line in fusion_lines[:5]:
    print(" ", line)

print("\n=== Pallas HLO (Custom-call inspection) ===")
pallas_lines = [
    line for line in hlo_pallas.splitlines() if "custom-call(" in line or "%add" in line
]
for line in pallas_lines[:5]:
    print(" ", line)

## 14. Summary & Profiling Methodology Checklist

When profiling JAX & Pallas kernels on TPU:
1. **Always warm up** before capturing traces to avoid polluting measurements with one-time compilation time.
2. **Isolate trace windows**: never capture multiple variants sharing the same kernel body in one trace window (XLA can merge modules under the same name).
3. **Separate Host vs Device**: wall-clock `time.perf_counter` measures Python dispatch overhead; read `dur` from `ph == "X"` events for pure TPU execution.
4. **Fix custom-call reporting**: attach `cost_estimate` to `pallas_call` so profilers can calculate memory bandwidth and TFLOPS.
5. **Inspect `.compile().as_text()`**: use post-compilation HLO to verify whether XLA actually merged operations or left them as separate kernels.